In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 设置随机种子以确保可重复性
np.random.seed(42)

# 生成1000条样本
n_samples = 1000
n_train = 800  # 训练集大小
n_test = 200   # 测试集大小

# 生成特征数据
def generate_diabetes_data(n_samples):
    data = []

    for i in range(1, n_samples + 1):
        # 基本特征
        age = np.random.randint(20, 80)
        gender = np.random.randint(0, 2)  # 0=女, 1=男

        # BMI (与糖尿病风险相关)
        base_bmi = np.random.normal(25, 4)
        bmi = max(18, min(45, base_bmi))

        # 医疗指标
        glucose = np.random.normal(100, 30)
        blood_pressure = np.random.normal(80, 12)
        skin_thickness = np.random.normal(25, 10)
        insulin = np.random.normal(120, 60)
        hba1c = np.random.normal(5.5, 1.2)

        # 生活习惯
        exercise_hours = np.random.exponential(3)
        diet_score = np.random.randint(1, 11)
        smoking = np.random.binomial(1, 0.3)
        alcohol = np.random.choice([0, 1, 2], p=[0.4, 0.4, 0.2])
        family_history = np.random.binomial(1, 0.25)

        # 计算糖尿病风险分数（真实关系）
        risk_score = (
            0.05 * max(0, age - 40) +  # 年龄风险
            0.1 * max(0, bmi - 25) +    # BMI风险
            0.15 * max(0, glucose - 100) / 20 +  # 血糖风险
            0.08 * max(0, hba1c - 5.7) +  # HbA1c风险
            0.3 * family_history +  # 家族史风险
            0.1 * smoking -  # 吸烟风险
            0.05 * min(exercise_hours, 10) -  # 运动保护
            0.03 * diet_score +  # 饮食保护（注意：这里应该是减号，但原代码是加号）
            0.1 * (alcohol == 2)  # 经常饮酒风险
        )

        # 添加随机噪声
        risk_score += np.random.normal(0, 0.3)

        # 生成糖尿病标签（概率随风险分数增加）
        diabetes_prob = 1 / (1 + np.exp(-risk_score))
        diabetes = 1 if diabetes_prob > 0.5 else 0

        # 调整阳性样本比例（约为35%）
        if diabetes == 1 and np.random.random() < 0.4:
            diabetes = 0
        elif diabetes == 0 and np.random.random() < 0.15:
            diabetes = 1

        data.append([
            i, age, gender, round(bmi, 1), round(max(70, min(200, glucose))),
            round(max(60, min(120, blood_pressure))),
            round(max(0, min(50, skin_thickness))) if np.random.random() > 0.15 else None,
            round(max(0, min(300, insulin))) if np.random.random() > 0.15 else None,
            round(max(4.0, min(10.0, hba1c)), 1),
            round(min(15, exercise_hours), 1),
            diet_score, smoking, alcohol, family_history, diabetes
        ])

    return data

# 生成数据
all_data = generate_diabetes_data(n_samples)

# 转换为DataFrame
columns = [
    'PatientID', 'Age', 'Gender', 'BMI', 'Glucose', 'BloodPressure',
    'SkinThickness', 'Insulin', 'HbA1c', 'ExerciseHours', 'DietScore',
    'Smoking', 'AlcoholConsumption', 'FamilyHistory', 'Diabetes'
]

df = pd.DataFrame(all_data, columns=columns)

# 分割训练集和测试集
train_df = df.iloc[:n_train].copy()
test_df = df.iloc[n_train:n_train + n_test].copy()

# 创建测试集（不含Diabetes列）
test_df_features = test_df.drop('Diabetes', axis=1)

# 保存到CSV文件
train_df.to_csv('train_diabetes.csv', index=False)
test_df_features.to_csv('test_diabetes.csv', index=False)

# 显示统计信息
print("训练集信息:")
print(f"样本数: {len(train_df)}")
print(f"糖尿病比例: {train_df['Diabetes'].mean():.2%}")
print("\n测试集信息:")
print(f"样本数: {len(test_df_features)}")
print(f"真实糖尿病比例: {test_df['Diabetes'].mean():.2%}")

print("\n前5行训练数据:")
print(train_df.head())
print("\n前5行测试数据:")
print(test_df_features.head())

训练集信息:
样本数: 800
糖尿病比例: 48.00%

测试集信息:
样本数: 200
真实糖尿病比例: 51.50%

前5行训练数据:
   PatientID  Age  Gender   BMI  Glucose  BloodPressure  SkinThickness  \
0          1   58       1  20.6      110             83           35.0   
1          2   66       1  26.3      102             63           20.0   
2          3   79       0  29.5      111             75            NaN   
3          4   76       1  28.5       70             61           33.0   
4          5   30       0  29.1      128             70            NaN   

   Insulin  HbA1c  ExerciseHours  DietScore  Smoking  AlcoholConsumption  \
0     85.0    4.9            0.6         10        0                   1   
1    127.0    4.1            0.2          4        1                   1   
2    154.0    4.7            1.7          6        0                   0   
3     88.0    4.0            2.3          9        0                   0   
4    140.0    6.7            1.2          7        0                   0   

   FamilyHistory  Diabete